Tentative Plan:


1.   Collect 1–3 public Gen Z/slang datasets.
2.   Normalize them into one input-output format.
3.   Fine-tune BART.
4.   Build 2 baselines.
5.   Create your own 100–200 sentence test set.
6.   Evaluate with human judgments on aspects like: meaning preserved, sounds Gen Z, sounds natural



## STEP 1

In [2]:
!pip -q install datasets pandas pyarrow

In [3]:
import pandas as pd
from datasets import load_dataset, Dataset

pd.set_option("display.max_colwidth", 200)

In [4]:
# Load the three Hugging Face datasets

# 1) Main slang lexicon / glossary dataset
ds_mlb = load_dataset("MLBtrio/genz-slang-dataset")

# 2) Paired sentence rewrite dataset
ds_pairs_1k = load_dataset("Programmer-RD-AI/genz-slang-pairs-1k")

# 3) Small paired translation dataset
ds_sherry = load_dataset("thesherrycode/gen-z-slangs-translation")

print(ds_mlb)
print(ds_pairs_1k)
print(ds_sherry)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

all_slangs.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/1779 [00:00<?, ? examples/s]

README.md: 0.00B [00:00, ?B/s]

genz_dataset.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/1005 [00:00<?, ? examples/s]

gen_z_slangs_translation.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/140 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['Slang', 'Description', 'Example', 'Context'],
        num_rows: 1779
    })
})
DatasetDict({
    train: Dataset({
        features: ['normal', 'gen_z'],
        num_rows: 1005
    })
})
DatasetDict({
    train: Dataset({
        features: ['Plain English', 'Gen-Z Slang'],
        num_rows: 140
    })
})


In [5]:
# Inspect schemas
print("MLBtrio columns:", ds_mlb["train"].column_names)
print("Pairs-1k columns:", ds_pairs_1k["train"].column_names)
print("Sherry columns:", ds_sherry["train"].column_names)

print("\nSample from MLBtrio:")
print(ds_mlb["train"][0])

print("\nSample from Pairs-1k:")
print(ds_pairs_1k["train"][0])

print("\nSample from Sherry:")
print(ds_sherry["train"][0])

MLBtrio columns: ['Slang', 'Description', 'Example', 'Context']
Pairs-1k columns: ['normal', 'gen_z']
Sherry columns: ['Plain English', 'Gen-Z Slang']

Sample from MLBtrio:
{'Slang': 'W', 'Description': 'Shorthand for win', 'Example': 'Got the job today, big W!', 'Context': 'Typically used in conversations to celebrate success, achievements, or positive outcomes'}

Sample from Pairs-1k:
{'normal': "I'm really tired today, I think I need some rest.", 'gen_z': "I'm totally drained today, need to catch some Z's."}

Sample from Sherry:
{'Plain English': 'That’s amazing!', 'Gen-Z Slang': 'That’s lit!'}


In [6]:
# Normalize the two parallel datasets into one format, one schema
#   source | plain | gen_z

pairs_1k_df = pd.DataFrame(ds_pairs_1k["train"])
pairs_1k_df = pairs_1k_df.rename(columns={
    "normal": "plain",
    "gen_z": "gen_z"
})
pairs_1k_df["source"] = "Programmer-RD-AI/genz-slang-pairs-1k"

sherry_df = pd.DataFrame(ds_sherry["train"])
sherry_df = sherry_df.rename(columns={
    "Plain English": "plain",
    "Gen-Z Slang": "gen_z"
})
sherry_df["source"] = "thesherrycode/gen-z-slangs-translation"

parallel_df = pd.concat(
    [
        pairs_1k_df[["source", "plain", "gen_z"]],
        sherry_df[["source", "plain", "gen_z"]],
    ],
    ignore_index=True
)

# basic cleanup
parallel_df = parallel_df.dropna()
parallel_df["plain"] = parallel_df["plain"].astype(str).str.strip()
parallel_df["gen_z"] = parallel_df["gen_z"].astype(str).str.strip()

# remove exact duplicates
parallel_df = parallel_df.drop_duplicates(subset=["plain", "gen_z"]).reset_index(drop=True)

print("Combined parallel dataset shape:", parallel_df.shape)
parallel_df.head(20)

Combined parallel dataset shape: (1144, 3)


,source,plain,gen_z
0,Programmer-RD-AI/genz-slang-pairs-1k,"I'm really tired today, I think I need some rest.","I'm totally drained today, need to catch some Z's."
1,Programmer-RD-AI/genz-slang-pairs-1k,"I'm really tired today, I just want to relax at home.","I'm hella drained today, just wanna chill at home."
2,Programmer-RD-AI/genz-slang-pairs-1k,I'm really excited for the concert tonight.,I'm so hype for the concert tonight.
3,Programmer-RD-AI/genz-slang-pairs-1k,"I'm really tired today, I think I need some coffee.","I'm so drained today, I gotta get me some caffeine."
4,Programmer-RD-AI/genz-slang-pairs-1k,I'm really looking forward to the weekend.,"I'm so hyped for the weekend, can't wait to turn up!"
5,Programmer-RD-AI/genz-slang-pairs-1k,I'm really tired today and just want to relax at home.,I'm mad tired today and just wanna chill at home.
6,Programmer-RD-AI/genz-slang-pairs-1k,I'm just hanging out with my friends tonight.,I'm just vibing with the squad tonight.
7,Programmer-RD-AI/genz-slang-pairs-1k,I'm going to grab some coffee before work.,"I'm tryna hit up some coffee before I clock in, fr."
8,Programmer-RD-AI/genz-slang-pairs-1k,"Hey, did you see the new movie last night? It was really good.","Yo, peep the new flick last night? It was lit."
9,Programmer-RD-AI/genz-slang-pairs-1k,I'm just chilling at home and watching some TV.,"Just vibing at home, Netflix and chillin'."


In [7]:
# Normalize the MLBtrio slang lexicon separately
mlb_df = pd.DataFrame(ds_mlb["train"])

# Normalize names if they exist exactly as expected
# Expected columns are: Slang, Description, Example, Context

expected_cols = ["Slang", "Description", "Example", "Context"]
missing = [c for c in expected_cols if c not in mlb_df.columns]
if missing:
    print("Warning: missing expected MLBtrio columns:", missing)

mlb_lexicon_df = mlb_df.copy()
mlb_lexicon_df["source"] = "MLBtrio/genz-slang-dataset"

# keep the most useful columns if present
keep_cols = [c for c in ["source", "Slang", "Description", "Example", "Context"] if c in mlb_lexicon_df.columns]
mlb_lexicon_df = mlb_lexicon_df[keep_cols].dropna(subset=["Slang"]).reset_index(drop=True)

print("MLB lexicon shape:", mlb_lexicon_df.shape)
mlb_lexicon_df.head(20)

MLB lexicon shape: (1779, 5)


,source,Slang,Description,Example,Context
0,MLBtrio/genz-slang-dataset,W,Shorthand for win,"Got the job today, big W!","Typically used in conversations to celebrate success, achievements, or positive outcomes"
1,MLBtrio/genz-slang-dataset,L,Shorthand for loss/losing,"I forgot my wallet at home, that’s an L.","Often used when referring to a failure or mishap, either personally or generally"
2,MLBtrio/genz-slang-dataset,L+ratio,Response to a comment or action on the internet that is particularly bad.,Your tweet got 5 likes and 100 replies calling you out. L + ratio.,Popularized on social media platforms to signify that someone not only failed but also had their failure amplified through backlash.
3,MLBtrio/genz-slang-dataset,Dank,excellent or of very high quality,That meme is so dank!,Commonly used in internet slang to refer to memes or humor that are edgy or particularly funny.
4,MLBtrio/genz-slang-dataset,Cheugy,Derogatory term for Millennials. Used when millennials are perceived to be excessively attempting to be trendy or stylish.,"That phrase is so cheugy, no one says that anymore.","Used to refer to things that were once popular but are now considered passé, usually in a millennial vs. Gen Z context."
5,MLBtrio/genz-slang-dataset,TFW,That feeling when,TFW you finish a big project and can finally relax,"Often paired with an image or caption to convey an emotion or experience, mostly used in memes."
6,MLBtrio/genz-slang-dataset,Woke,being politically aware,He’s really woke about climate change.,"Originally used in activist circles, it has since been co-opted and sometimes used sarcastically."
7,MLBtrio/genz-slang-dataset,Bop,An excellent song or album.,This new track is a total bop!,"Commonly used in reference to music, especially upbeat songs that are hard to resist dancing to."
8,MLBtrio/genz-slang-dataset,G.O.A.T,The greatest of all time,Michael Jordan is the GOAT of basketball.,Frequently used in sports but has expanded to any field where someone is considered the best.
9,MLBtrio/genz-slang-dataset,Smol,"Something that is small, and in most cases exceptionally adorable.","Look at that smol puppy, it’s adorable!","Typically used when describing pets, babies, or anything tiny and endearing."


In [8]:
# Create a model-ready Hugging Face Dataset object
hf_parallel = Dataset.from_pandas(parallel_df)

print(hf_parallel)
print(hf_parallel[0])

Dataset({
    features: ['source', 'plain', 'gen_z'],
    num_rows: 1144
})
{'source': 'Programmer-RD-AI/genz-slang-pairs-1k', 'plain': "I'm really tired today, I think I need some rest.", 'gen_z': "I'm totally drained today, need to catch some Z's."}


In [9]:
print("Total paired examples:", len(parallel_df))
print("Examples by source:")
print(parallel_df["source"].value_counts())

print("\nRandom paired samples:")
display(parallel_df.sample(min(10, len(parallel_df)), random_state=42))

print("\nRandom slang lexicon samples:")
display(mlb_lexicon_df.sample(min(10, len(mlb_lexicon_df)), random_state=42))

Total paired examples: 1144
Examples by source:
source
Programmer-RD-AI/genz-slang-pairs-1k      1005
thesherrycode/gen-z-slangs-translation     139
Name: count, dtype: int64

Random paired samples:


,source,plain,gen_z
218,Programmer-RD-AI/genz-slang-pairs-1k,I'm so tired after studying all night.,I'm hella drained after pulling an all-nighter.
809,Programmer-RD-AI/genz-slang-pairs-1k,"Hey, I just got back from the store and grabbed some snacks.","Yo, just hit the store and stacked up on snacks, fr fr!"
501,Programmer-RD-AI/genz-slang-pairs-1k,"Hey, I just finished my homework and I’m feeling pretty good.","Yo, just peeped my homework done and I’m vibing hard."
649,Programmer-RD-AI/genz-slang-pairs-1k,I'm just going to chill at home all day today.,"I'm just gonna veg out at home all day today, no cap."
323,Programmer-RD-AI/genz-slang-pairs-1k,I'm just going to relax and watch some videos this afternoon.,I'm just gonna chill and scroll through vids today.
506,Programmer-RD-AI/genz-slang-pairs-1k,I'm feeling really tired today after staying up late yesterday.,I'm mad drained today 'cause I stayed up all night yesterday.
1005,thesherrycode/gen-z-slangs-translation,That’s amazing!,That’s lit!
107,Programmer-RD-AI/genz-slang-pairs-1k,I'm just going to relax at home and watch some TV.,I'm just chillin' at home and vibing with some Netflix.
731,Programmer-RD-AI/genz-slang-pairs-1k,I'm just going to relax and watch some videos later.,"I'm just chillin' and vibin' with some vids later, fr."
170,Programmer-RD-AI/genz-slang-pairs-1k,I'm just going to grab some coffee before heading to work.,I'm finna sip some java before I dip to work.



Random slang lexicon samples:


,source,Slang,Description,Example,Context
1192,MLBtrio/genz-slang-dataset,NR,Nice roll,"You got double sixes, NR!",A compliment used in dice games when someone gets a good result.
426,MLBtrio/genz-slang-dataset,BAK,Back at keyboard,"BRB, BAK in 5.",Used to inform someone that you are back at your computer after being away.
1474,MLBtrio/genz-slang-dataset,SWALK,Sealed with a loving kiss,The letter had SWALK written on the envelope.,"A more affectionate version of SWAK, often used in love letters."
765,MLBtrio/genz-slang-dataset,FOMO,Fear of missing out,"I have such FOMO, everyone’s going to that concert except me.",Refers to the feeling of anxiety or regret over missing an event or experience.
485,MLBtrio/genz-slang-dataset,BIOYN,Blow it out your nose,"You’re being ridiculous, BIOYN.",A less harsh but still dismissive insult.
479,MLBtrio/genz-slang-dataset,BIF,Before I forget,"BIF, don’t forget the meeting at 3!",Used to quickly mention something before it slips the person’s mind.
109,MLBtrio/genz-slang-dataset,ick,used to express disgust at something unpleasant or offensive.,"He talks with his mouth full, that gave me the ick.",Frequently used in dating scenarios to describe a moment that turns someone off.
774,MLBtrio/genz-slang-dataset,FU,Freak you,"You messed up my plans, FU!",A harsh or angry way to express frustration or anger towards someone.
1431,MLBtrio/genz-slang-dataset,SMAZED,Smoky haze,The sky was SMAZED after the fireworks.,"Describes a smoky or hazy atmosphere, often caused by smoke."
1188,MLBtrio/genz-slang-dataset,NOYB,None of your business,What’s going on with you? NOYB!,A blunt way to tell someone to stay out of your personal matters.


In [10]:
# Save clean CSVs so GitHub/deployment is easier later
parallel_df.to_csv("genz_parallel_combined.csv", index=False)
mlb_lexicon_df.to_csv("genz_slang_lexicon.csv", index=False)

print("Saved:")
print("- genz_parallel_combined.csv")
print("- genz_slang_lexicon.csv")

Saved:
- genz_parallel_combined.csv
- genz_slang_lexicon.csv


In [11]:
# Build a lightweight lexical substitution map
slang_terms = mlb_lexicon_df["Slang"].dropna().astype(str).str.strip().unique().tolist()

print("Number of unique slang terms:", len(slang_terms))
print("Sample slang terms:", slang_terms[:50])

Number of unique slang terms: 1588
Sample slang terms: ['W', 'L', 'L+ratio', 'Dank', 'Cheugy', 'TFW', 'Woke', 'Bop', 'G.O.A.T', 'Smol', 'Fam', 'Glow up', 'Stan', 'Ghosting', 'Salty', 'Sip tea', 'Drip', 'Iykyk', 'Rent free', 'Catch these hands', 'Drag', 'Bussin’', 'Snatched', 'Cancel culture', 'Ffs', 'e-boy', 'e-girl', 'Big yikes', 'Finna', 'cap', 'High-key', 'Simp', 'Camp', 'Snack', 'Take several seats', 'Sheesh', 'Hits different', 'Bet', 'Periodt', 'Finesse', 'I’m weak', 'Main character', 'Sis', 'Sending me', 'Gaup', 'This ain’t it, chief', 'Extra', 'Clapback', '@ me', 'Asl']


## Step 2

In [12]:
!pip -q install transformers accelerate sentencepiece scikit-learn

import pandas as pd
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer

In [13]:
# Reload the cleaned paired dataset
# Otherwise load from CSV
try:
    parallel_df
    print("Using parallel_df already in memory.")
except NameError:
    parallel_df = pd.read_csv("genz_parallel_combined.csv")
    print("Loaded parallel_df from CSV.")

print(parallel_df.shape)
parallel_df.head()

Using parallel_df already in memory.
(1144, 3)


,source,plain,gen_z
0,Programmer-RD-AI/genz-slang-pairs-1k,"I'm really tired today, I think I need some rest.","I'm totally drained today, need to catch some Z's."
1,Programmer-RD-AI/genz-slang-pairs-1k,"I'm really tired today, I just want to relax at home.","I'm hella drained today, just wanna chill at home."
2,Programmer-RD-AI/genz-slang-pairs-1k,I'm really excited for the concert tonight.,I'm so hype for the concert tonight.
3,Programmer-RD-AI/genz-slang-pairs-1k,"I'm really tired today, I think I need some coffee.","I'm so drained today, I gotta get me some caffeine."
4,Programmer-RD-AI/genz-slang-pairs-1k,I'm really looking forward to the weekend.,"I'm so hyped for the weekend, can't wait to turn up!"


In [14]:
parallel_df = parallel_df.copy()

parallel_df["plain"] = parallel_df["plain"].astype(str).str.strip()
parallel_df["gen_z"] = parallel_df["gen_z"].astype(str).str.strip()

# Remove empty rows if any slipped through
parallel_df = parallel_df[
    (parallel_df["plain"] != "") &
    (parallel_df["gen_z"] != "")
].reset_index(drop=True)

print("Cleaned shape:", parallel_df.shape)

Cleaned shape: (1144, 3)


In [15]:
# Add model input prompt, frame task clearly for BART
parallel_df["input_text"] = "translate to Gen Z: " + parallel_df["plain"]
parallel_df["target_text"] = parallel_df["gen_z"]

parallel_df[["input_text", "target_text"]].head(10)

,input_text,target_text
0,"translate to Gen Z: I'm really tired today, I think I need some rest.","I'm totally drained today, need to catch some Z's."
1,"translate to Gen Z: I'm really tired today, I just want to relax at home.","I'm hella drained today, just wanna chill at home."
2,translate to Gen Z: I'm really excited for the concert tonight.,I'm so hype for the concert tonight.
3,"translate to Gen Z: I'm really tired today, I think I need some coffee.","I'm so drained today, I gotta get me some caffeine."
4,translate to Gen Z: I'm really looking forward to the weekend.,"I'm so hyped for the weekend, can't wait to turn up!"
5,translate to Gen Z: I'm really tired today and just want to relax at home.,I'm mad tired today and just wanna chill at home.
6,translate to Gen Z: I'm just hanging out with my friends tonight.,I'm just vibing with the squad tonight.
7,translate to Gen Z: I'm going to grab some coffee before work.,"I'm tryna hit up some coffee before I clock in, fr."
8,"translate to Gen Z: Hey, did you see the new movie last night? It was really good.","Yo, peep the new flick last night? It was lit."
9,translate to Gen Z: I'm just chilling at home and watching some TV.,"Just vibing at home, Netflix and chillin'."


In [16]:
# Train/Validation split, 90/10 for now
train_df, val_df = train_test_split(
    parallel_df[["source", "plain", "gen_z", "input_text", "target_text"]],
    test_size=0.10,
    random_state=42,
    shuffle=True
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print("Train size:", len(train_df))
print("Validation size:", len(val_df))

Train size: 1029
Validation size: 115


In [17]:
# Convert to Hugging Face DatasetDict
train_ds = Dataset.from_pandas(train_df, preserve_index=False)
val_ds = Dataset.from_pandas(val_df, preserve_index=False)

dataset_dict = DatasetDict({
    "train": train_ds,
    "validation": val_ds
})

print(dataset_dict)
print(dataset_dict["train"][0])

DatasetDict({
    train: Dataset({
        features: ['source', 'plain', 'gen_z', 'input_text', 'target_text'],
        num_rows: 1029
    })
    validation: Dataset({
        features: ['source', 'plain', 'gen_z', 'input_text', 'target_text'],
        num_rows: 115
    })
})
{'source': 'Programmer-RD-AI/genz-slang-pairs-1k', 'plain': "I can't believe how fun the concert was last night!", 'gen_z': 'Yo, the concert last night was straight fire!', 'input_text': "translate to Gen Z: I can't believe how fun the concert was last night!", 'target_text': 'Yo, the concert last night was straight fire!'}


In [18]:
# Load BART tokenizer, start from facebook/bart-base, adjust in later steps
checkpoint = "facebook/bart-base"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

print("Tokenizer loaded:", checkpoint)

config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer loaded: facebook/bart-base


In [19]:
# tokenization settings
max_input_length = 64
max_target_length = 64

In [20]:
def preprocess_function(examples):
    model_inputs = tokenizer(
        examples["input_text"],
        max_length=max_input_length,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        text_target=examples["target_text"],
        max_length=max_target_length,
        truncation=True,
        padding="max_length"
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_datasets = dataset_dict.map(
    preprocess_function,
    batched=True,
    remove_columns=dataset_dict["train"].column_names
)

print(tokenized_datasets)
print(tokenized_datasets["train"][0].keys())

Map:   0%|          | 0/1029 [00:00<?, ? examples/s]

Map:   0%|          | 0/115 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 1029
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 115
    })
})
dict_keys(['input_ids', 'attention_mask', 'labels'])


In [21]:
# inspect a decoded example
sample_idx = 0

raw_example = dataset_dict["train"][sample_idx]
tok_example = tokenized_datasets["train"][sample_idx]

print("RAW INPUT:")
print(raw_example["input_text"])
print("\nRAW TARGET:")
print(raw_example["target_text"])

print("\nTOKENIZED INPUT DECODED:")
print(tokenizer.decode(tok_example["input_ids"], skip_special_tokens=True))

label_ids = [x for x in tok_example["labels"] if x != tokenizer.pad_token_id]
print("\nTOKENIZED TARGET DECODED:")
print(tokenizer.decode(label_ids, skip_special_tokens=True))

RAW INPUT:
translate to Gen Z: I can't believe how fun the concert was last night!

RAW TARGET:
Yo, the concert last night was straight fire!

TOKENIZED INPUT DECODED:
translate to Gen Z: I can't believe how fun the concert was last night!

TOKENIZED TARGET DECODED:
Yo, the concert last night was straight fire!


In [22]:
# Save processed artifacts for later
train_df.to_csv("genz_train_split.csv", index=False)
val_df.to_csv("genz_val_split.csv", index=False)

tokenized_datasets.save_to_disk("genz_tokenized_bart_dataset")

print("Saved:")
print("- genz_train_split.csv")
print("- genz_val_split.csv")
print("- genz_tokenized_bart_dataset/")

Saving the dataset (0/1 shards):   0%|          | 0/1029 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/115 [00:00<?, ? examples/s]

Saved:
- genz_train_split.csv
- genz_val_split.csv
- genz_tokenized_bart_dataset/


## Step 3

In [23]:
!pip -q install transformers accelerate evaluate rouge_score
import re
import random
import numpy as np
import evaluate

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.5 MB/s eta 0:00:00


In [24]:
import torch
print(torch.cuda.get_device_name(0))

NVIDIA A100-SXM4-80GB


In [25]:
# Setting up rule-based slang baseline, very simple
# A lightweight starter dictionary.
# Can expand this later using the MLB lexicon or custom mappings.
slang_map = {
    r"\bhello\b": "yo",
    r"\bhi\b": "yo",
    r"\bfriend\b": "bestie",
    r"\bfriends\b": "besties",
    r"\bvery\b": "lowkey",
    r"\breally\b": "actually",
    r"\bgreat\b": "fire",
    r"\bgood\b": "solid",
    r"\bbad\b": "mid",
    r"\bamazing\b": "bussin",
    r"\bexcellent\b": "elite",
    r"\bexcited\b": "hyped",
    r"\btired\b": "drained",
    r"\bembarrassing\b": "cringe",
    r"\blying\b": "capping",
    r"\blie\b": "cap",
    r"\bno lie\b": "no cap",
    r"\bfor real\b": "fr",
    r"\bof course\b": "bet",
    r"\bI agree\b": "facts",
    r"\bthat is a big win\b": "that's a W",
    r"\bthat is a loss\b": "that's an L",
}

def rule_based_genz(text):
    out = text

    # phrase-level replacements first
    for pattern, replacement in slang_map.items():
        out = re.sub(pattern, replacement, out, flags=re.IGNORECASE)

    # stylistic tweaks
    out = re.sub(r"\bI am\b", "I'm", out, flags=re.IGNORECASE)
    out = re.sub(r"\bdo not\b", "don't", out, flags=re.IGNORECASE)
    out = re.sub(r"\bcannot\b", "can't", out, flags=re.IGNORECASE)

    # optional Gen Z-ish intensifier injection
    if not out.endswith(("!", "?", ".")):
        out += "."

    if random.random() < 0.3:
        out = out.replace(".", " fr.")

    return out

In [26]:
# Quick test of the rule-based baseline
test_sentences = [
    "I am very excited for the concert tonight.",
    "My friend is really tired today.",
    "That was very embarrassing.",
    "I do not agree with that.",
    "This food is amazing."
]

for s in test_sentences:
    print("PLAIN: ", s)
    print("RULE:  ", rule_based_genz(s))
    print()

PLAIN:  I am very excited for the concert tonight.
RULE:   I'm lowkey hyped for the concert tonight fr.

PLAIN:  My friend is really tired today.
RULE:   My bestie is actually drained today fr.

PLAIN:  That was very embarrassing.
RULE:   That was lowkey cringe fr.

PLAIN:  I do not agree with that.
RULE:   I don't agree with that.

PLAIN:  This food is amazing.
RULE:   This food is bussin.



In [27]:
# Reload tokenizer and model
checkpoint = "facebook/bart-base"

tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSeq2SeqLM.from_pretrained(checkpoint)

print("Loaded model and tokenizer:", checkpoint)

model.safetensors:   0%|          | 0.00/558M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

Loaded model and tokenizer: facebook/bart-base


In [28]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

In [37]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./bart_genz_model",
    eval_strategy="no",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=50,
    learning_rate=5e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    num_train_epochs=4,
    predict_with_generate=False,
    bf16=True,
    fp16=False,
    save_total_limit=2,
    load_best_model_at_end=False,
    report_to="none"
)

In [38]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
)

trainer.train()

Step,Training Loss
50,0.218638
100,0.211711
150,0.205529
200,0.169583
250,0.167764
300,0.149708
350,0.139765
400,0.136580
450,0.125069
500,0.124512


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=516, training_loss=0.16337769784668618, metrics={'train_runtime': 42.8839, 'train_samples_per_second': 95.98, 'train_steps_per_second': 12.032, 'total_flos': 156854703882240.0, 'train_loss': 0.16337769784668618, 'epoch': 4.0})

In [39]:
# Save final model
save_dir = "./final_bart_genz_model"

trainer.save_model(save_dir)
tokenizer.save_pretrained(save_dir)

print(f"Saved model and tokenizer to {save_dir}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved model and tokenizer to ./final_bart_genz_model


In [40]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

def genz_translate(text, max_new_tokens=50):
    prompt = f"translate to Gen Z: {text}"
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=64
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=4,
            early_stopping=True
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [44]:
sample_df = val_df.sample(min(15, len(val_df)), random_state=42)

for _, row in sample_df.iterrows():
    source_text = row["plain"]
    ref_text = row["gen_z"]
    pred_text = genz_translate(source_text)

    print("PLAIN: ", source_text)
    print("GEN Z: ", ref_text)
    print("MODEL: ", pred_text)
    print("-" * 70)

PLAIN:  I can't believe how tired I am after a long day at work.
GEN Z:  Bruh, I'm mad wiped after a whole day grind.
MODEL:  I'm so drained after a long day at work, fr.
----------------------------------------------------------------------
PLAIN:  I'm just going to relax and watch some videos this afternoon.
GEN Z:  I'm just gonna chill and scroll through vids today.
MODEL:  I'm just chillin' and vibing with some vids later.
----------------------------------------------------------------------
PLAIN:  I think I need to take a break and relax for a bit.
GEN Z:  Bruh, I gotta vibe out and chill for a sec.
MODEL:  I gotta chill and veg out for a sec.
----------------------------------------------------------------------
PLAIN:  I’m sorry.
GEN Z:  My bad.
MODEL:  I’m mad.
----------------------------------------------------------------------
PLAIN:  Hey, do you want to go grab some coffee after school?
GEN Z:  Yo, wanna dip for some coffee after class?
MODEL:  Yo, wanna hit up some coff

## Step 4 (WIP)

In [45]:
# Compare rule-based baseline vs BART

# Make sure model is in eval mode
model.eval()

comparison_sample_df = val_df.sample(min(25, len(val_df)), random_state=42).reset_index(drop=True)

comparison_rows = []

for _, row in comparison_sample_df.iterrows():
    input_text = row["plain"]
    reference_text = row["gen_z"]

    rule_output = rule_based_genz(input_text)
    bart_output = genz_translate(input_text)

    comparison_rows.append({
        "input_text": input_text,
        "reference_text": reference_text,
        "rule_output": rule_output,
        "bart_output": bart_output
    })

comparison_df = pd.DataFrame(comparison_rows)
comparison_df.head(10)

,input_text,reference_text,rule_output,bart_output
0,I can't believe how tired I am after a long day at work.,"Bruh, I'm mad wiped after a whole day grind.",I can't believe how drained I'm after a long day at work.,"I'm so drained after a long day at work, fr."
1,I'm just going to relax and watch some videos this afternoon.,I'm just gonna chill and scroll through vids today.,I'm just going to relax and watch some videos this afternoon fr.,I'm just chillin' and vibing with some vids later.
2,I think I need to take a break and relax for a bit.,"Bruh, I gotta vibe out and chill for a sec.",I think I need to take a break and relax for a bit fr.,I gotta chill and veg out for a sec.
3,I’m sorry.,My bad.,I’m sorry fr.,I’m mad.
4,"Hey, do you want to go grab some coffee after school?","Yo, wanna dip for some coffee after class?","Hey, do you want to go grab some coffee after school?","Yo, wanna hit up some coffee after class?"
5,"I'm really tired today, so I think I'll just stay home and relax.","I'm super wiped today, so I'm just vibing at home and chilling.","I'm actually drained today, so I think I'll just stay home and relax.","I'm hella drained today, so I'm just gonna chill at home and vibe."
6,I'm just hanging out with my friends at the park today.,"Chillin' with my crew at the park today, no cap.",I'm just hanging out with my besties at the park today.,I'm just vibing with my squad at the park today.
7,I'm really tired after staying up all night studying.,I'm hella drained 'cause I pulled an all-nighter studying.,I'm actually drained after staying up all night studying fr.,I'm totally drained 'cause I pulled an all-nighter hitting the books.
8,I’m exhausted.,I’m beat.,I’m exhausted.,I’m drained.
9,I'm thinking about going to the mall later to hang out with my friends.,I'm vibing to hit the mall later and kick it with my crew.,I'm thinking about going to the mall later to hang out with my besties fr.,I'm vibing at the mall later to chill with my squad.


In [46]:
for _, row in comparison_df.head(10).iterrows():
    print("INPUT:      ", row["input_text"])
    print("REFERENCE:  ", row["reference_text"])
    print("RULE:       ", row["rule_output"])
    print("BART:       ", row["bart_output"])
    print("-" * 80)

INPUT:       I can't believe how tired I am after a long day at work.
REFERENCE:   Bruh, I'm mad wiped after a whole day grind.
RULE:        I can't believe how drained I'm after a long day at work.
BART:        I'm so drained after a long day at work, fr.
--------------------------------------------------------------------------------
INPUT:       I'm just going to relax and watch some videos this afternoon.
REFERENCE:   I'm just gonna chill and scroll through vids today.
RULE:        I'm just going to relax and watch some videos this afternoon fr.
BART:        I'm just chillin' and vibing with some vids later.
--------------------------------------------------------------------------------
INPUT:       I think I need to take a break and relax for a bit.
REFERENCE:   Bruh, I gotta vibe out and chill for a sec.
RULE:        I think I need to take a break and relax for a bit fr.
BART:        I gotta chill and veg out for a sec.
-----------------------------------------------------------

In [ ]:
# Manual evaluation template
manual_eval_df = comparison_df.copy()

manual_eval_df["rule_meaning"] = None
manual_eval_df["rule_style"] = None
manual_eval_df["rule_fluency"] = None

manual_eval_df["bart_meaning"] = None
manual_eval_df["bart_style"] = None
manual_eval_df["bart_fluency"] = None

manual_eval_df["overall_preference"] = None
manual_eval_df["notes"] = None

manual_eval_df.head(10)

In [ ]:
manual_eval_df.to_csv("manual_eval_template.csv", index=False)
print("Saved manual_eval_template.csv")

In [ ]:
# Manual eval sample
eval_sample_df = comparison_df.head(10).copy().reset_index(drop=True)

for i, row in eval_sample_df.iterrows():
    print(f"\nExample {i+1}")
    print("INPUT:      ", row["input_text"])
    print("REFERENCE:  ", row["reference_text"])
    print("RULE:       ", row["rule_output"])
    print("BART:       ", row["bart_output"])
    print("-" * 80)

In [ ]:
# Example manual scoring format
# Replace these with your actual judgments

eval_sample_df["rule_meaning"] = [3, 4, 4, 1, 3, 4, 4, 4, 4, 3]
eval_sample_df["rule_style"] = [2, 2, 2, 1, 2, 2, 2, 2, 1, 2]
eval_sample_df["rule_fluency"] = [3, 3, 3, 1, 3, 3, 3, 3, 2, 3]

eval_sample_df["bart_meaning"] = [5, 4, 4, 1, 4, 5, 4, 5, 4, 3]
eval_sample_df["bart_style"] = [4, 4, 3, 1, 4, 4, 4, 4, 2, 4]
eval_sample_df["bart_fluency"] = [5, 5, 4, 1, 5, 5, 5, 5, 4, 4]

eval_sample_df["overall_preference"] = [
    "bart", "bart", "bart", "tie", "bart",
    "bart", "bart", "bart", "bart", "bart"
]

eval_sample_df["notes"] = [
    "BART preserved tone better",
    "BART more natural",
    "Both okay, BART more fluent",
    "Both failed meaning",
    "BART sounds cleaner",
    "BART stronger style",
    "BART more natural",
    "BART best overall",
    "BART weaker style but okay",
    "BART slightly drifts meaning"
]

eval_sample_df

## Step 5 Eval Metrics (on val and baseline for now)

In [47]:
# Compare automatic metrics (BART)
import evaluate

rouge = evaluate.load("rouge")

bart_predictions = []
references = []

for _, row in val_df.iterrows():
    pred = genz_translate(row["plain"])
    bart_predictions.append(pred.strip())
    references.append(row["gen_z"].strip())

bart_results = rouge.compute(
    predictions=bart_predictions,
    references=references,
    use_stemmer=True
)

bart_results

{'rouge1': np.float64(0.7024857688545588),
 'rouge2': np.float64(0.5075897924684507),
 'rougeL': np.float64(0.6903681169735898),
 'rougeLsum': np.float64(0.6891297305735111)}

In [48]:
# Rule baseline metrics
rule_predictions = []

for _, row in val_df.iterrows():
    pred = rule_based_genz(row["plain"])
    rule_predictions.append(pred.strip())

rule_results = rouge.compute(
    predictions=rule_predictions,
    references=references,
    use_stemmer=True
)

rule_results

{'rouge1': np.float64(0.5361381409332163),
 'rouge2': np.float64(0.298790307456131),
 'rougeL': np.float64(0.5279878187215437),
 'rougeLsum': np.float64(0.528129411788413)}

In [49]:
metrics_df = pd.DataFrame([
    {
        "model": "rule_based",
        "rouge1": rule_results["rouge1"],
        "rouge2": rule_results["rouge2"],
        "rougeL": rule_results["rougeL"]
    },
    {
        "model": "bart",
        "rouge1": bart_results["rouge1"],
        "rouge2": bart_results["rouge2"],
        "rougeL": bart_results["rougeL"]
    }
])

metrics_df

,model,rouge1,rouge2,rougeL
0,rule_based,0.536138,0.29879,0.527988
1,bart,0.702486,0.50759,0.690368


In [1]:
!pip -q install bert-score

from bert_score import score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.7 MB/s eta 0:00:00


In [2]:
bart_preds = bart_predictions
refs = references

P, R, F1 = score(bart_preds, refs, lang="en", verbose=True)

bertscore_results = {
    "precision": P.mean().item(),
    "recall": R.mean().item(),
    "f1": F1.mean().item()
}

bertscore_results

NameError: name 'bart_predictions' is not defined

In [ ]:
slang_vocab = set([s.lower() for s in slang_terms])

def slang_score(text):
    tokens = text.lower().split()
    if len(tokens) == 0:
        return 0
    count = sum(1 for t in tokens if t in slang_vocab)
    return count / len(tokens)

bart_slang_scores = [slang_score(p) for p in bart_predictions]
rule_slang_scores = [slang_score(p) for p in rule_predictions]

print("BART slang score:", sum(bart_slang_scores) / len(bart_slang_scores))
print("RULE slang score:", sum(rule_slang_scores) / len(rule_slang_scores))

In [ ]:
def length_ratio(preds, inputs):
    return sum(len(p.split()) / len(i.split()) for p, i in zip(preds, inputs)) / len(preds)

bart_len = length_ratio(bart_predictions, val_df["plain"])
rule_len = length_ratio(rule_predictions, val_df["plain"])

print("BART length ratio:", bart_len)
print("RULE length ratio:", rule_len)

## Step 6: Create Test Data (WIP)